# Week 8 — Systems, Auth & Supply Chain: trust the artifact, not the name

**Lesson plan:** [`../weeks/week-08.md`](../weeks/week-08.md)
**Reading:** the XZ-utils backdoor (2024) writeup + a dependency-confusion primer
(Birsan, 2021).

Pure Python, no lab target. The last classical-security week, and the one closest
to how systems actually get breached in 2025: not by breaking crypto, but by
**trusting the wrong artifact** — a package that isn't yours, a token you didn't
really sign, a name that resolved to an attacker.

> ### The one idea
> Software trusts *names* — a package name, an `alg` field, a version string — when
> it should trust *artifacts*: a pinned hash, a signature you verified under a key
> you chose. Every attack this week substitutes an artifact behind a name the
> victim trusted. The fix is always the same shape: **pin the artifact, don't trust
> the name.**

## 1 · ⚠️ Dependency confusion — the public registry shadows your internal one

Your build installs `acme-auth`, an internal package. The installer resolves the
name against your internal registry **and** the public one (PyPI/npm), and — like
real installers by default — picks the **highest version**. An attacker publishes
`acme-auth` publicly with version 9.9.9. Your build now installs *their* code under
your trusted internal name. This is Alex Birsan's 2021 attack that breached Apple,
Microsoft, and dozens more.

In [1]:
# Two registries. The attacker has squatted your internal name on the public one.
internal_registry = {"acme-auth": "1.4.2", "acme-utils": "2.1.0"}
public_registry   = {"acme-auth": "9.9.9", "requests": "2.31.0"}   # 9.9.9 = attacker

def resolve_highest_version(pkg):
    """A naive resolver: consider both registries, pick the highest version,
    IGNORE the source. This is essentially the default behavior that made
    dependency confusion work at scale."""
    candidates = []
    if pkg in internal_registry:
        candidates.append(("internal", internal_registry[pkg]))
    if pkg in public_registry:
        candidates.append(("public", public_registry[pkg]))
    return max(candidates, key=lambda c: tuple(int(x) for x in c[1].split(".")))

source, version = resolve_highest_version("acme-auth")
print(f"installing 'acme-auth' -> resolved to the {source} registry, v{version}")
print(f"attacker's package won: {source == 'public'}")
print("""
Your internal name, the attacker's code, installed by your own CI. Nobody typo'd
anything; the resolver trusted the NAME and picked by version. 'acme-auth' meant
two different artifacts, and the wrong one had a bigger number.""")

installing 'acme-auth' -> resolved to the public registry, v9.9.9
attacker's package won: True

Your internal name, the attacker's code, installed by your own CI. Nobody typo'd
anything; the resolver trusted the NAME and picked by version. 'acme-auth' meant
two different artifacts, and the wrong one had a bigger number.


## 2 · The fix — pin the artifact (lockfile + digest + source)

Don't resolve by name and version. Resolve against a **lockfile** that records the
exact hash and the source you vetted. Now "acme-auth" isn't a name to look
up — it's a specific artifact you've already approved, and anything else is
rejected.

In [2]:
import hashlib

# The lockfile: the exact artifact you vetted, by hash and source.
LOCKFILE = {
    "acme-auth": {"version": "1.4.2", "sha256": "aa11bb22cc", "source": "internal"},
}

def resolve_pinned(pkg, candidate_hash, candidate_source):
    want = LOCKFILE.get(pkg)
    if not want:
        return False, "package not in lockfile — refuse to install"
    if candidate_source != want["source"]:
        return False, f"source mismatch: got {candidate_source!r}, pinned {want['source']!r}"
    if candidate_hash != want["sha256"]:
        return False, "hash mismatch — artifact is not the one we vetted"
    return True, "ok — exact artifact matches the lockfile"

print("attacker's public package:", resolve_pinned("acme-auth", "evil99ff", "public"))
print("legitimate internal build:", resolve_pinned("acme-auth", "aa11bb22cc", "internal"))
print("""
The pin makes the name irrelevant: only the artifact you approved installs. This is
what a lockfile with hashes (pip --require-hashes, package-lock.json) and scoped
internal registries buy you. Trust the artifact, not the name.""")

attacker's public package: (False, "source mismatch: got 'public', pinned 'internal'")
legitimate internal build: (True, 'ok — exact artifact matches the lockfile')

The pin makes the name irrelevant: only the artifact you approved installs. This is
what a lockfile with hashes (pip --require-hashes, package-lock.json) and scoped
internal registries buy you. Trust the artifact, not the name.


## 3 · SBOM — you can't pin what you can't see

Pinning requires *knowing your dependencies*, including the transitive ones you
never chose directly. A **Software Bill of Materials** is that inventory. Its
security value: when a CVE (or a backdoor like XZ-utils) drops, you can answer "are
we affected?" in seconds instead of weeks.

In [3]:
# A tiny dependency graph -> flattened SBOM. Real tools (syft, cyclonedx) do this
# across the whole transitive tree.
dep_graph = {
    "our-app":    ["acme-auth", "requests"],
    "acme-auth":  ["pyjwt", "xz-utils"],
    "requests":   ["urllib3", "certifi"],
    "pyjwt":      [],
    "xz-utils":   [],          # the 2024 backdoor lived HERE, deep in the tree
    "urllib3":    [], "certifi": [],
}

def build_sbom(root, graph, seen=None):
    seen = seen if seen is not None else set()
    for dep in graph.get(root, []):
        if dep not in seen:
            seen.add(dep)
            build_sbom(dep, graph, seen)
    return sorted(seen)

sbom = build_sbom("our-app", dep_graph)
print("SBOM (all transitive dependencies of our-app):")
for pkg in sbom:
    print(f"  - {pkg}")

# Incident response: "the xz-utils backdoor just dropped. Are we exposed?"
KNOWN_BAD = {"xz-utils"}
affected = [p for p in sbom if p in KNOWN_BAD]
print(f"\nCVE lookup against the SBOM -> affected: {affected}")
print("""
'xz-utils' is a TRANSITIVE dependency — nobody chose it directly; acme-auth pulled
it in. Without an SBOM you'd be grepping build logs across every repo for days.
With one, the query is instant. You can't pin, patch, or even panic about what you
can't enumerate.""")

SBOM (all transitive dependencies of our-app):
  - acme-auth
  - certifi
  - pyjwt
  - requests
  - urllib3
  - xz-utils

CVE lookup against the SBOM -> affected: ['xz-utils']

'xz-utils' is a TRANSITIVE dependency — nobody chose it directly; acme-auth pulled
it in. Without an SBOM you'd be grepping build logs across every repo for days.
With one, the query is instant. You can't pin, patch, or even panic about what you
can't enumerate.


## 4 · ⚠️ Auth: the `alg=none` token forgery

Same disease, different limb. A signed session token (JWT) carries a header saying
which algorithm signed it. A shocking number of verifiers **trust that header** —
including a value of `"none"`, meaning "no signature." An attacker sets `alg=none`,
edits the payload to `role: admin`, and sends *no signature at all*. The server
trusts the token's self-description and lets them in.

In [4]:
import hmac, json, base64

def b64(data):
    return base64.urlsafe_b64encode(data).rstrip(b"=").decode()

def b64d(s):
    return base64.urlsafe_b64decode(s + "=" * (-len(s) % 4))

SECRET = b"server-signing-key"

def make_token(payload):
    header = b64(json.dumps({"alg": "HS256", "typ": "JWT"}).encode())
    body = b64(json.dumps(payload).encode())
    sig = b64(hmac.new(SECRET, f"{header}.{body}".encode(), hashlib.sha256).digest())
    return f"{header}.{body}.{sig}"

def verify_insecure(token):
    """Trusts the token's OWN alg header — including 'none'. The bug."""
    h, p, sig = token.split(".")
    if json.loads(b64d(h)).get("alg") == "none":
        return json.loads(b64d(p))                    # no signature check at all
    expected = b64(hmac.new(SECRET, f"{h}.{p}".encode(), hashlib.sha256).digest())
    return json.loads(b64d(p)) if hmac.compare_digest(sig, expected) else None

legit = make_token({"user": "alice", "role": "user"})
print("legit token verifies as:", verify_insecure(legit))

# FORGE: alg=none, role=admin, empty signature.
h = b64(json.dumps({"alg": "none", "typ": "JWT"}).encode())
p = b64(json.dumps({"user": "alice", "role": "admin"}).encode())
forged = f"{h}.{p}."                                   # note the empty signature
print("alg=none forgery verifies as:", verify_insecure(forged))
print("  -> privilege escalated to admin, no signing key required")

legit token verifies as: {'user': 'alice', 'role': 'user'}
alg=none forgery verifies as: {'user': 'alice', 'role': 'admin'}
  -> privilege escalated to admin, no signing key required


In [5]:
def verify_secure(token, expected_alg="HS256"):
    """Pin the algorithm SERVER-SIDE. Never trust the token's own header."""
    h, p, sig = token.split(".")
    if json.loads(b64d(h)).get("alg") != expected_alg:
        return None                                    # reject anything but our alg
    expected = b64(hmac.new(SECRET, f"{h}.{p}".encode(), hashlib.sha256).digest())
    return json.loads(b64d(p)) if hmac.compare_digest(sig, expected) else None

print("with server-side algorithm pinning:")
print("  alg=none forgery:", verify_secure(forged))
print("  legit token     :", verify_secure(legit))
print("""
The forgery trusted the token to describe its own verification. The fix is the same
as sections 1-2: don't trust the NAME/header the artifact gives you — pin what YOU
expect (the algorithm) and verify against it. Trust the artifact under YOUR key,
not the token's self-description.""")

with server-side algorithm pinning:
  alg=none forgery: None
  legit token     : {'user': 'alice', 'role': 'user'}

The forgery trusted the token to describe its own verification. The fix is the same
as sections 1-2: don't trust the NAME/header the artifact gives you — pin what YOU
expect (the algorithm) and verify against it. Trust the artifact under YOUR key,
not the token's self-description.


## 5 · The scorecard view — trust the artifact, not the name

| Attack | The trusted name | The substituted artifact | The pin that fixes it |
|---|---|---|---|
| Dependency confusion | package name | attacker's public package | lockfile + hash + source scope |
| Transitive backdoor (XZ) | "it's just a dependency" | compromised upstream code | SBOM + provenance + pinning |
| `alg=none` JWT | the token's `alg` header | an unsigned token | server-side algorithm pinning |

> Every 2025-era supply-chain and auth breach in this table is the same move:
> **the victim trusted a name, and an attacker put a different artifact behind it.**
> The defense is monotonous and it works: pin the artifact (hash, source,
> algorithm), verify under a key/list *you* control, and inventory what you depend
> on so you can pin it at all. Guarantee (axis 2): *only the exact vetted artifact
> is trusted* — which a name-based lookup can never promise.

## 6 · Your studio deliverable — and HW 2

In `week08/`:

1. **Dependency confusion** — implement the naive highest-version resolver, show the
   public package winning, then defeat it with a lockfile+hash+source pin.
2. **SBOM** — build the transitive dependency list for a small graph; answer a
   "are we affected by CVE-X?" query against it. Explain why transitive deps are the
   dangerous ones.
3. **`alg=none` forgery** — forge an admin token against a header-trusting verifier;
   fix it with server-side algorithm pinning. (Bonus: the weak-HMAC-secret variant —
   brute-force a bad signing key.)
4. **Control Scorecard** — for each, name the trusted *name*, the substituted
   *artifact*, and the *pin* that closes it. State the guarantee pinning provides.
